# ECON 5140: Applied Econometrics
## Homework 3: Causal Inference — Potential Outcomes & Matching

**Covers:** Lesson 7 (Potential Outcomes & DAGs) & Lesson 9 (Matching for Causal Inference)

---

# PART 1: ANALYTICAL PROBLEMS

### Problem 1: Potential Outcomes and Causal Estimands

Consider the following data from 8 individuals. D = 1 if they received job training; D = 0 if not. Y(1) and Y(0) are potential outcomes (earnings in thousands).

| i | Y(1) | Y(0) | D | Y_obs |
|---|------|------|---|-------|
| 1 | 45   | 35   | 1 | 45    |
| 2 | 50   | 42   | 1 | 50    |
| 3 | 38   | 40   | 0 | 40    |
| 4 | 55   | 48   | 1 | 55    |
| 5 | 42   | 38   | 0 | 38    |
| 6 | 48   | 45   | 1 | 48    |
| 7 | 40   | 44   | 0 | 44    |
| 8 | 52   | 46   | 0 | 46    |

**Questions:**

a) Calculate ATE, ATT, and ATU.

b) Calculate the Simple Difference in Outcomes (SDO) = E[Y|D=1] − E[Y|D=0]. Is SDO equal to ATE? Why or why not?

c) Decompose SDO into ATE + selection bias + HTE bias. Verify: SDO = ATE + (E[Y(0)|D=1] − E[Y(0)|D=0]) + p×(ATT − ATU).

d) If treatment were randomly assigned, what would happen to the SDO? Explain.

### Problem 2: DAGs — Confounding and Colliders

**Part (a)** Consider the DAG: D ← X → Y and D → Y, where D = treatment, Y = outcome, X = observed covariate.

- What is X called? Why does a naive comparison of D and Y (without controlling for X) produce bias?
- What adjustment would you make to estimate the causal effect of D on Y?

**Part (b)** Consider the DAG: D → X ← Y, where X is a collider.

- Are D and Y independent in the population? Explain.
- What happens if we condition on X? Why is this problematic?
- Give a real-world example of a collider (similar to the "admitted to elite university" example from lecture).

### Problem 3: Exact vs Approximate Matching

a) What is exact matching? When does it work well? What is the "curse of dimensionality" and why does it limit exact matching?

b) What is approximate (distance-based) matching? Name two distance metrics. How does nearest-neighbor matching work?

### Problem 4: Inverse Probability Weighting (IPW) — Numerical Example

Suppose we observe the following data. D = 1 if received job training, D = 0 if not. Y = earnings (thousands). The propensity score e(X) = Pr(D=1|X) has been estimated for each unit.

| i | D | Y  | e(X) |
|---|---|-----|------|
| 1 | 1 | 52 | 0.6  |
| 2 | 1 | 48 | 0.8  |
| 3 | 0 | 42 | 0.3  |
| 4 | 0 | 38 | 0.2  |
| 5 | 1 | 55 | 0.7  |
| 6 | 0 | 45 | 0.4  |

**Questions:**

a) **ATE weights:** For each unit i, the IPW weight for ATE is w_i = D_i/e(X_i) + (1−D_i)/(1−e(X_i)). Compute the weight for each of the 6 units. Fill in the table:

| i | D | Y  | e(X) | w_ATE |
|---|---|-----|------|-------|
| 1 | 1 | 52 | 0.6  | ?     |
| 2 | 1 | 48 | 0.8  | ?     |
| ... |   |    |      |       |

b) **ATT weights:** For ATT, treated units get weight 1; control units get weight w_i = e(X_i)/(1−e(X_i)). Compute the ATT weight for each unit. Fill in the table:

| i | D | Y  | e(X) | w_ATT |
|---|---|-----|------|-------|
| 1 | 1 | 52 | 0.6  | ?     |
| ... |   |    |      |       |

c) **ATE estimate:** Compute the IPW estimator for ATE:


d) **ATT estimate:** Compute the IPW estimator for ATT:



e) **Interpretation:** Which unit has the largest ATE weight? Why? (Hint: "Units unlikely to receive their observed treatment get larger weight.")

# PART 2: CODING PROBLEMS

## Problem 5: Propensity Score Model (Coding)

Using **simulated data**, build a propensity score model for causal inference. The setup: job training (D) and earnings (Y), with covariates X1, X2, and education. Treatment is not randomly assigned—higher X1, X2, and education increase the probability of receiving training. The true causal effect of training on earnings is 10.

**Your tasks:**

1. **Generate simulated data** (or use the provided code below) with D, Y, X1, X2, education.

2. **Estimate the propensity score** e(X) = Pr(D=1|X) using logistic regression of D on X1, X2, and education. Print the model summary (coefficients, interpretation).

3. **Add fitted propensity scores** to the dataset.

4. **Check overlap:** Plot the propensity score distributions for treated vs control (KDE or histogram). Comment on whether overlap is sufficient.

5. **Compute ATE via IPW** using the estimated propensity scores. Compare to the naive ATE and the true effect (10). Briefly interpret.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings('ignore')

np.random.seed(123)

# Generate dataset
n = 400
X1 = np.random.normal(0, 1, n)
X2 = np.clip(np.random.normal(0, 1, n), -2, 2)
education = np.random.choice([0, 1, 2], n, p=[0.4, 0.35, 0.25])

logit_p = -0.5 + 0.4*X1 + 0.4*X2 + 0.3*education
ps_true = 1 / (1 + np.exp(-logit_p))
D = (np.random.uniform(0, 1, n) < ps_true).astype(int)

Y = 50 + 10*D + 5*X1 + 5*X2 + 3*education + np.random.normal(0, 5, n)

df = pd.DataFrame({'X1': X1, 'X2': X2, 'education': education, 'D': D, 'Y': Y})

print("Dataset summary:")
print(f"  n = {len(df)}, Treated: {D.sum()}, Control: {n - D.sum()}")
print(df.head(10))

Dataset summary:
  n = 400, Treated: 179, Control: 221
         X1        X2  education  D          Y
0 -1.085631  1.534090          0  0  56.431975
1  0.997345 -0.529914          1  0  59.038149
2  0.282978 -0.490972          1  1  63.978546
3 -1.506295 -1.309165          0  0  34.873068
4 -0.578600 -0.008660          1  0  47.433150
5  1.651437  0.976813          0  1  71.823104
6 -2.426679 -1.751070          0  0  32.957026
7 -0.428913 -0.665857          2  0  47.114497
8  1.265936  0.035941          0  0  56.757740
9 -0.866740  0.850103          2  0  47.178667


In [2]:
# YOUR CODE for Problem 5
# 1. Estimate propensity score (logistic regression)
# 2. Add fitted ps to df
# 3. Plot overlap (KDE or histogram)
# 4. Compute ATE via IPW
# 5. Compare naive ATE, IPW ATE, and true effect (10)

